# Session 07 — Drone Human Detection with HOG (Baseline)

This standalone Google Colab notebook reproduces the **traditional HOG + SVM people detector** baseline to compare against the YOLO + ByteTrack notebook. **HOG is NOT YOLO and does not do identity tracking.** The detector often misses small people in high-angle drone footage.

**How to use:** Run all cells. Upload `test_video.mp4` from the bundled `03_Source_Videos` folder when prompted. No paid software or external model download is needed.

In [ ]:
import cv2, numpy as np, subprocess, os
from IPython.display import Video, display
try:
    from google.colab import files
except ImportError:
    files=None
print('OpenCV version:', cv2.__version__)
assert files is not None, 'Upload flow is intended for Google Colab.'


In [ ]:
uploaded=files.upload()
if not uploaded: raise RuntimeError('Please upload the original drone MP4 file.')
input_video=next(iter(uploaded))
cap=cv2.VideoCapture(input_video)
assert cap.isOpened(), f'Cannot open video {input_video}'
print('Loaded:', input_video)
cap.release()


In [ ]:
# Independent HOG baseline with adjustable settings.
MAX_FRAMES = 150         # set None to process the whole video
MAX_WIDTH = 800          # speed vs. target size: small people are very difficult for HOG
HOG_SCORE_THRESHOLD = 0.20

hog=cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())
cap=cv2.VideoCapture(input_video)
assert cap.isOpened(), 'Could not read source video.'
fps=cap.get(cv2.CAP_PROP_FPS) or 24
orig_w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));orig_h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
scale=min(1,MAX_WIDTH/orig_w)
w=max(1,int(orig_w*scale));h=max(1,int(orig_h*scale))
raw_out='drone_hog_output_raw.mp4';final_out='drone_hog_output.mp4'
writer=cv2.VideoWriter(raw_out,cv2.VideoWriter_fourcc(*'mp4v'),fps,(w,h))
assert writer.isOpened(), 'Could not create output video.'
counts=[]
while True:
    if MAX_FRAMES is not None and len(counts)>=MAX_FRAMES: break
    ok,frame=cap.read()
    if not ok: break
    if scale!=1: frame=cv2.resize(frame,(w,h))
    detections,weights=hog.detectMultiScale(frame,winStride=(8,8),padding=(8,8),scale=1.05)
    valid=[]
    for (x,y,bw,bh),score in zip(detections,weights):
        if float(score)>=HOG_SCORE_THRESHOLD:
            valid.append((x,y,bw,bh))
    for x,y,bw,bh in valid:
        cv2.rectangle(frame,(x,y),(x+bw,y+bh),(0,220,80),2)
    counts.append(len(valid))
    cv2.rectangle(frame,(0,0,w,70),(30,28,25),-1)
    cv2.putText(frame,'DRONE | HOG baseline',(12,27),cv2.FONT_HERSHEY_SIMPLEX,.75,(255,255,255),2)
    cv2.putText(frame,f'People detected now: {len(valid)}',(12,56),cv2.FONT_HERSHEY_SIMPLEX,.68,(100,255,100),2)
    writer.write(frame)
cap.release();writer.release()
if not counts: raise RuntimeError('No frames were processed.')
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',raw_out,'-c:v','libx264','-pix_fmt','yuv420p','-movflags','+faststart',final_out],check=True)
print('Processed frames:',len(counts),'| Max detections/frame:',max(counts),'| Mean:',round(float(np.mean(counts)),2))
print('A zero count does NOT mean there are no people; HOG may fail on aerial viewpoints.')
display(Video(final_out,embed=True,width=760))


## Instructor activity — HOG versus YOLO

Using the same drone source video, compare the HOG baseline output with the YOLO + ByteTrack output in `04_Result_Videos`. Students should answer:
1. Why does HOG often report zero people despite visible crowds?
2. Does lowering the detection score always improve precision?
3. What is the difference between **detections per frame**, **assigned tracking IDs**, and **unique people**?
4. Why can a detector still miss people during occlusion or from a high camera altitude?

**Important evaluation limitation:** Existing HOG/YOLO MP4 results in this archive were generated in earlier separate runs and do not necessarily represent the same frame window or parameters. To make a quantitative comparison, rerun both notebooks using the same source footage and frame range, and create manual ground-truth annotations.